### **Mel Scale, Mel filterbank**
---

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
import time
import numpy as np
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio

sys.path.insert(0, os.path.abspath(".."))
from dsp.filters import convolve, fft_convolve
from dsp.signal_generator import make_time, create_sin
from dsp.features import hz_to_mel, mel_to_hz

**P1 (ทฤษฎี + โค้ด + หู) Mel scale — อ่าน Wikipedia "Mel scale" ย่อหน้าแรกและหัวข้อ Formula**
  - **ทำ:**
    - สร้าง `notebooks/08_Perception.ipynb` ใช้ `sr = 8000`
    - `pair_low` = 200 Hz 0.5 วินาที ต่อด้วย 300 Hz 0.5 วินาที (แอมพลิจูด 0.5, ใช้ `np.concatenate`) → ยาว 8000 sample
    - `pair_high` = 3000 Hz ต่อด้วย 3100 Hz แบบเดียวกัน
    - ฟังทั้งสองคู่ แล้วเขียน markdown 1 ประโยค: คู่ไหนได้ยินเสียงเปลี่ยนชัดกว่า (ทั้งคู่ห่างกัน 100 Hz)
    > จากการทดลองฟังเสียงแล้วพบว่า `pair_low` ฟังแล้วได้ยินเสียงการเปลี่ยนแปลงที่ชัดเจนกว่า `pair_high`
    - สร้าง `dsp/features.py` แล้วเขียน `hz_to_mel(f)` และ `mel_to_hz(m)` พร้อม docstring
    - ทดสอบ: print `hz_to_mel` ของ 0, 700, 1000 Hz
    - ทดสอบไป-กลับและเทียบ librosa ด้วย `f_test = np.array([0, 440, 4000])` (ใช้ `np.allclose` ทั้งสองอัน)
    - print ระยะ mel ของคู่ต่ำ (300 Hz − 200 Hz) และคู่สูง (3100 Hz − 3000 Hz)
  - **เครื่องมือใหม่:** `10 ** 2` → `100` / `librosa.hz_to_mel(700, htk=True)` → ≈781.17 (`htk=True` = สูตรเดียวกับเรา)
  - **ใบ้:** $m = 2595 \log_{10}(1 + f/700)$ และ $f = 700\,(10^{m/2595} - 1)$ / ใช้ `np.log10` ฟังก์ชันจึงรับ array ได้ / สูตรตั้งให้ 1000 Hz ≈ 1000 mel
  - **ตอบ:**
    1. `hz_to_mel(700)` ได้กี่ mel
    > `hz_to_mel(700)` จะมีค่าเท่ากับ 781.17 mel
    2. ระยะ mel ของคู่ต่ำได้กี่ mel
    > ระยะ mel ของคู่ต่ำมีค่าเท่ากับ 118.74 mel
    3. ระยะ mel ของคู่สูงได้กี่ mel
    > ระยะ mel ของคู่สูงมีค่าเท่ากับ 30.05 mel.
  - **เสร็จเมื่อ:** 0, 700, 1000 Hz ได้ 0, ≈781.17, ≈999.99 mel / `allclose` ได้ True ทั้งสองอัน / คู่ต่ำ ≈118.7 mel คู่สูง ≈30.1 mel / ตอบครบ

In [2]:
sr_p1 = 8000
t_p1 = make_time(0, 0.5, sr_p1)

pair_low = np.concatenate([create_sin(0.5, 200, t_p1), create_sin(0.5, 300, t_p1)])
pair_high = np.concatenate([create_sin(0.5, 3000, t_p1), create_sin(0.5, 3100, t_p1)])

In [3]:
# Listen pair_low 
Audio(pair_low, rate=sr_p1)

In [4]:
# Listen pair_high
Audio(pair_high, rate=sr_p1)

In [5]:
# Test program dsp/features using hz_to_me
F = [0, 700, 1000]
print("Test Program.")
for f in F:
    print(f"Change {f} Hz to {round(hz_to_mel(f),2)} mel.")

print(" ")
# Test with f_test = np.array([0, 440, 4000])
f_test = np.array([0, 440, 4000])
mel_test = hz_to_mel(f_test)

print(f"Convert from Hz to mel : {mel_test}")
print(f"Check hz_to_mel() with librosa.hz_to_mel() : {np.allclose(mel_test, librosa.hz_to_mel(f_test, htk=True))}")
print(f"Convert from mel to Hz : {mel_to_hz(mel_test)}")
print(f"Check mel_to_Hz() with librosa.hz_to_mel() : {np.allclose(mel_to_hz(mel_test), librosa.mel_to_hz(mel_test, htk=True))} \n")

print(f"pair_low in Mel scale : {round(hz_to_mel(300)-hz_to_mel(200),2)} mel.")
print(f"pair_high in Mel scale : {round(hz_to_mel(3100)-hz_to_mel(3000),2)} mel.")
     

Test Program.
Change 0 Hz to 0.0 mel.
Change 700 Hz to 781.17 mel.
Change 1000 Hz to 999.99 mel.
 
Convert from Hz to mel : [   0.          549.63867538 2146.06452751]
Check hz_to_mel() with librosa.hz_to_mel() : True
Convert from mel to Hz : [   0.  440. 4000.]
Check mel_to_Hz() with librosa.hz_to_mel() : True 

pair_low in Mel scale : 118.74 mel.
pair_high in Mel scale : 30.05 mel.
